# AM — Aula prática 01: classificação supervisionada

**Disciplina:** Aprendizado de Máquina · **Ferramentas:** Python, `scikit-learn`, `pandas`, `matplotlib`/`seaborn`

## Objetivos de aprendizagem (formativos)

Este laboratório tem caráter **formativo**: não há **uma única resposta correta** para todos os números e gráficos. O valor está em **experimentar**, **registrar** o que você fez (código, figuras, comentários breves) e **relacionar** com a teoria da disciplina.

- Explorar dados com **EDA** e raciocinar sobre **pré-processamento**.
- Montar **pipelines** (`StandardScaler` + classificador) e treinar **árvore de decisão**, **SVM** e **Naive Bayes** (Gaussiano).
- **Explorar hiperparâmetros** e relacionar com *overfitting*, generalização e intuição de cada modelo.
- Usar **validação cruzada** com `Pipeline` e entender **vazamento de informação** (*data leakage*).
- Opcionalmente aprofundar com **ROC**, **métricas** além da acurácia e **importância de variáveis**.

**Trabalho em dupla** é incentivado para discutir resultados; cada pessoa deve **manter seu próprio notebook** com o código que executou e suas observações.

## Contexto e uso responsável dos dados

O conjunto **Breast Cancer Wisconsin** (`load_breast_cancer`) é **clássico no ensino de ML**: atributos numéricos, rótulo binário (maligno vs benigno). Ele serve para **aprender métodos e pipelines**, não para substituir **diagnóstico clínico**. Há limitações de **época**, **população** e **contexto de coleta**; não extrapole conclusões para aplicações reais sem estudo específico.

## Dataset (técnico)

**Breast Cancer Wisconsin** — classificação binária, apenas atributos numéricos (`sklearn.datasets.load_breast_cancer`).

## Como usar este notebook

As **partes 1–4** são o **núcleo** da experimentação; a **parte 5** é **exploração livre** (sala ou casa). As células de **código vazio** são o **seu espaço**: implemente, teste e refine o **seu** pipeline. Há **uma** célula de **demonstração** logo após os imports — execute como referência; o trabalho formativo está em **reproduzir, variar e interpretar** nas demais células.


## 0. Configuração do ambiente

Execute a célula abaixo. Se faltar algum pacote: `pip install scikit-learn pandas matplotlib seaborn`.


In [1]:
# Imports — amplie com outros módulos do sklearn conforme os enunciados
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
    precision_recall_fscore_support,
)

# Em exploração, os avisos do sklearn podem ser pedagógicos (convergência etc.).
# Descomente a linha seguinte se quiser suprimir avisos de usuário.
# warnings.filterwarnings("ignore", category=UserWarning)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")
rng_seed = 42


### Conexão rápida com a teoria

| Abordagem | Lembrete |
|-----------|----------|
| **Árvore de decisão** | Cortes nos eixos das features; a **escala** afeta **quais** partições são escolhidas primeiro (e portanto o modelo), de forma diferente de SVM. |
| **SVM (kernel RBF)** | Fronteira não linear; **escalonamento** costuma ser essencial. `C` e `gamma` regulam **margem** vs **ajuste** aos dados. |
| **Naive Bayes Gaussiano** | Hipótese de **independência condicional** entre features; bom **baseline** probabilístico. |

Relacione o que você observar com **viés**, **variância** e **capacidade** do modelo vistos na aula teórica.


In [2]:
# Demonstração — pipeline mínimo (referência). Depois construa o seu nas atividades.
# Objetivo: mostrar fit → predict → acurácia num fluxo único. Não substitui as Partes 1–4.

data_demo = load_breast_cancer()
X_d, y_d = data_demo.data, data_demo.target
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_d, y_d, test_size=0.25, random_state=rng_seed, stratify=y_d
)
scaler_d = StandardScaler()
X_train_sd = scaler_d.fit_transform(X_train_d)
X_test_sd = scaler_d.transform(X_test_d)

clf_demo = SVC(kernel="rbf", random_state=rng_seed)
clf_demo.fit(X_train_sd, y_train_d)
acc_demo = accuracy_score(y_test_d, clf_demo.predict(X_test_sd))
print("Demo: acurácia (SVM RBF + dados escalonados):", round(acc_demo, 4))


Demo: acurácia (SVM RBF + dados escalonados): 0.979


---

## Parte 1 — Carga dos dados e EDA

Use as **células de código vazias** desta parte para **sua** exploração; acrescente células se precisar de mais espaço.

### Atividade 1.1 — Primeiro contato
1. Carregue o dataset com `load_breast_cancer()` e monte um `pandas.DataFrame` com as features e a coluna alvo (rótulo).
2. Informe: número de linhas, número de features, nomes das classes (`target_names` do objeto retornado pelo `load_breast_cancer`).
3. Verifique se há valores ausentes (`NaN`) nas colunas (por exemplo com `DataFrame.isna` somado por coluna ou no total).
4. Conte quantos exemplos há em cada classe e comente se a distribuição é aproximadamente **balanceada** ou não.

> **Enunciado:** você pode usar `sklearn.datasets.load_breast_cancer`, atributos `.data`, `.target`, `.feature_names`, `.target_names`, e `pandas`.


In [47]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)

X = pd.DataFrame(X)
y = pd.DataFrame(y)
print(f"Numero de linhas e colunas de X : {X.shape} e Y : {y.shape}\nTarget Name: {y.columns.values}")
X.isna()


Numero de linhas e colunas de X : (569, 30) e Y : (569, 1)
Target Name: ['target']


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
565,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
566,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
567,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [29]:

print(y.value_counts())

target
1         357
0         212
Name: count, dtype: int64


### Atividade 1.2 — Escalas e relações entre variáveis
1. Use estatísticas descritivas (`describe()` ou equivalente) e aponte **duas** features cuja **escala** (ordem de grandeza) seja claramente diferente.
2. Produza uma visualização da **matriz de correlação** entre as features (por exemplo com `seaborn.heatmap` ou `pandas.DataFrame.corr`). Comente brevemente se há grupos de variáveis muito correlacionadas.

> **Enunciado:** métodos úteis incluem `DataFrame.describe`, `DataFrame.corr`, `seaborn.heatmap`.


In [ ]:
X.describe()



,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [ ]:
X[["worst area","mean radius"]].corr()

,worst area,mean radius
worst area,1.000000,0.941082
mean radius,0.941082,1.000000


### Leitura rápida (conceito)

Na cópia do dataset distribuída pelo `scikit-learn`, costuma **não** haver valores faltantes — ainda assim, em projetos reais a checagem é obrigatória. Para imputação, veja `sklearn.impute.SimpleImputer` na documentação oficial.

---

## Parte 2 — Divisão dos dados, escalonamento e baseline

Aqui você monta o **pipeline** de pré-processamento e os três classificadores — use quantas células de código forem necessárias (incluindo células novas).

### Atividade 2.1 — Treino / teste e `StandardScaler`
1. Defina `X` (matriz de features) e `y` (vetor de rótulos).
2. Divida em treino e teste com `sklearn.model_selection.train_test_split`, com `test_size=0.25`, `random_state=42` e `stratify=y`.
3. Ajuste um `StandardScaler` **apenas** com o conjunto de treino (`fit` no treino; `transform` em treino e teste). Explique em uma frase por que não se deve usar o conjunto de teste para calcular média e desvio usados na padronização.

> **Enunciado:** `sklearn.preprocessing.StandardScaler`, `train_test_split`.


In [50]:
X_train, X_test, y_train, y_test = train_test_split(X,y ,random_state=42,shuffle=True,test_size=0.2)
print(f"X: Train {X_train.shape} Test {X_test.shape}\nY: Train {y_train.shape} Test {y_test.shape} ")

X: Train (455, 30) Test (114, 30)
Y: Train (455, 1) Test (114, 1) 


### Atividade 2.2 — Três classificadores (primeira versão)

Treine **separadamente** (no treino escalonado, avalie no teste escalonado):

| Modelo | Classe |
|--------|--------|
| Árvore de decisão | `sklearn.tree.DecisionTreeClassifier` |
| SVM | `sklearn.svm.SVC` |
| Naive Bayes | `sklearn.naive_bayes.GaussianNB` |

Para cada um: instancie o estimador, chame `fit`, obtenha predições no teste, calcule **acurácia** e exiba `sklearn.metrics.classification_report`. Use `random_state=42` na árvore e no SVM onde aplicável; para o kernel do SVM, use por exemplo RBF (`kernel="rbf"`).

Em **markdown** (célula de reflexão da Parte 2.2), responda: a **árvore** depende da escala das features da mesma forma que o SVM? Por quê?

> **Enunciado:** `fit`, `predict`, `accuracy_score`, `classification_report`.


#### Reflexão em texto — Parte 2.2 (árvore vs escala)





### Atividade 2.3 — Matriz de confusão

Escolha **um** dos três modelos treinados acima e exiba a **matriz de confusão** no conjunto de teste. Você pode usar `sklearn.metrics.ConfusionMatrixDisplay.from_predictions` ou combinar `confusion_matrix` com um gráfico.

Explique o significado de **falso positivo** e **falso negativo** neste problema (maligno vs benigno): qual tipo de erro costuma ser mais grave em um cenário de triagem médica? *(Reflexão em markdown na célula dedicada abaixo.)*

> **Enunciado:** `ConfusionMatrixDisplay`, `confusion_matrix`.


#### Reflexão em texto — Parte 2.3 (interpretação)





---

## Parte 3 — Hiperparâmetros (trilha principal de experimentação)

Nesta parte você **explora** o efeito de hiperparâmetros: varie valores, registre resultados (tabelas ou gráficos) e **conecte** com capacidade do modelo e *overfitting*. O objetivo é **ver o que muda**, não acertar um único número.

### Atividade 3.1 — Árvore de decisão
Varie de forma explícita **pelo menos dois** hiperparâmetros da árvore (exemplos: `max_depth`, `min_samples_leaf`, `min_samples_split`, `max_leaf_nodes`). Para cada configuração (ou para um conjunto reduzido de combinações), avalie o desempenho usando **validação cruzada** no conjunto de treino (por exemplo `cross_val_score` com `Pipeline` contendo apenas o classificador na árvore, ou validação estratificada). **Não use o conjunto de teste** para escolher hiperparâmetros.

Apresente uma **tabela ou gráfico** (por exemplo: acurácia média vs `max_depth`) e indique qual configuração você escolheria e por quê.

> **Enunciado:** `DecisionTreeClassifier`, `sklearn.model_selection.cross_val_score`, `StratifiedKFold`; opcionalmente `sklearn.model_selection.validation_curve` para uma grade unidimensional de valores.


#### Reflexão em texto — Parte 3.1 (conclusão da árvore)





### Atividade 3.2 — SVM
Construa uma **grade pequena** de valores para os hiperparâmetros `C` e `gamma` do `SVC` com `kernel="rbf"` (por exemplo 3×3 valores ou menos). Use `sklearn.model_selection.GridSearchCV` com validação cruzada estratificada (`cv` adequado, `random_state` onde couber) e `Pipeline` com `StandardScaler` + `SVC`, treinando **somente** nos dados de treino da Parte 2.

Reporte a **melhor combinação** (`best_params_`) e a acurácia de validação associada (`best_score_` ou equivalente). Em seguida, avalie o melhor modelo no conjunto de **teste** (métrica à sua escolha, por exemplo acurácia ou F1).

Na **reflexão em texto** correspondente, comente o que você observou ao variar `C` e `gamma` (por exemplo, tendência a *overfitting* ou subajuste) e relacione com a intuição desses hiperparâmetros no SVM com kernel RBF.

> **Enunciado:** `GridSearchCV`, `Pipeline`, `StandardScaler`, `SVC`. Consulte a documentação do `GridSearchCV` para o dicionário `param_grid` com nomes de etapas do pipeline.


#### Reflexão em texto — Parte 3.2 (interpretação)





### Atividade 3.3 — Naive Bayes Gaussiano
O `GaussianNB` tem menos botões que árvore e SVM, mas possui por exemplo o hiperparâmetro `var_smoothing`. Experimente **pelo menos três** valores de `var_smoothing` em escala logarítmica (ex.: `np.logspace(...)`) usando validação cruzada no **treino**, compare as acurácias médias e escolha um valor. Opcional: avalie o modelo escolhido no teste.

> **Enunciado:** `GaussianNB`, `cross_val_score` ou `GridSearchCV`.


---

## Parte 4 — Validação cruzada global e *data leakage*

**Por que esta parte depois da 3?** Na Parte 3 você ajustou hiperparâmetros usando **apenas o treino** (ou subconjuntos dele) e reservou o **teste** para avaliação final pontual. Aqui o foco é outro: estimar desempenho com **validação cruzada no dataset completo** usando um **único `Pipeline` bem definido** (escalonador + modelo), útil para **comparar** com o holdout e para praticar o padrão correto sem vazamento. Não é redundância vã: é **dois papéis** — seleção de hiperparâmetros (Parte 3) vs **estimativa de generalização** com pipeline íntegro (Parte 4).

### Atividade 4.1 — `Pipeline` e `cross_val_score`
Com os dados **completos** `X` e `y`, monte um `Pipeline` com `StandardScaler` + um classificador (por exemplo o **melhor SVM** que você encontrou na Parte 3.2, ou um `SVC` com parâmetros fixos documentados). Use `cross_val_score` com `cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` e `scoring="accuracy"`.

Calcule média e desvio padrão das acurácias. Compare mentalmente com o desempenho no holdout da Parte 2.

> **Enunciado:** `sklearn.pipeline.Pipeline`, `cross_val_score`, `StratifiedKFold`.



### Atividade 4.2 — Discussão (*leakage*)

Explique por que o `StandardScaler` deve estar **dentro** do `Pipeline` quando se usa `cross_val_score` ou `GridSearchCV`, em vez de padronizar `X` inteiro antes.

> **Enunciado:** conceito de vazamento de informação entre treino e validação; documentação do `Pipeline`.


#### Reflexão em texto — Parte 4.2





---

## Parte 5 — Explorações extras (didáticas, para casa ou aprofundamento)

Escolha o que interessar à sua dupla ou ao seu ritmo; cada item é **independente**.

### Atividade 5.1 — Curva ROC e AUC
No conjunto de **teste**, escolha um modelo treinado no **treino escalonado** (por exemplo o melhor pipeline de SVM ou a árvore com hiperparâmetros escolhidos) e obtenha **probabilidades** ou **scores de decisão** compatíveis com a curva ROC (`predict_proba` ou `decision_function`, conforme o estimador). Trace a curva ROC e informe a **AUC** (`sklearn.metrics.roc_auc_score` ou `RocCurveDisplay`).

Discuta: a acurácia sozinha é suficiente neste problema?

> **Enunciado:** `roc_auc_score`, `RocCurveDisplay.from_predictions` ou `from_estimator`.


#### Reflexão em texto — Parte 5.1





### Atividade 5.2 — Importância das features (árvore)
Ajuste uma árvore (com hiperparâmetros que você julgar razoáveis) e utilize o atributo `feature_importances_`. Visualize as importâncias (por exemplo barras horizontais com nomes em `data.feature_names`). Quais atributos aparecem no topo? Isso condiz com o que você viu na matriz de correlação?

> **Enunciado:** `DecisionTreeClassifier.feature_importances_`, `matplotlib` ou `seaborn`.


### Atividade 5.3 — Comparação de métricas além da acurácia
Calcule no teste: **precisão**, **revocação** e **F1** para a classe positiva (defina qual classe é “positiva” no seu relatório). Use `sklearn.metrics.precision_recall_fscore_support` ou extraia de `classification_report`. Quando uma métrica pode ser mais informativa que a acurácia?

> **Enunciado:** métricas em `sklearn.metrics`.


### Atividade 5.4 — (Opcional) Curva de aprendizado
Use `sklearn.model_selection.learning_curve` com um pipeline (escalonador + classificador) para ver como a métrica evolui com o tamanho do conjunto de treino. O que sugere sobre viés e variância?

> **Enunciado:** `learning_curve`.


### Atividade 5.5 — (Opcional) Naive Bayes multinomial
Este dataset é numérico contínuo; o `MultinomialNB` é típico de contagens. Pesquise na documentação quando usar `MultinomialNB` vs `GaussianNB` e descreva um exemplo de problema para cada um.

#### Reflexão em texto — Parte 5.5





### Atividade 5.6 — (Opcional) Importância por permutação

A importância baseada em `feature_importances_` na árvore é **específica desse modelo**. Uma alternativa mais geral é medir quanto a métrica piora quando cada atributo é embaralhado aleatoriamente no conjunto de **teste** (ou validação). Use `sklearn.inspection.permutation_importance` com um pipeline já treinado (por exemplo SVM ou árvore) e compare mentalmente com o ranking da Parte 5.2.

> **Enunciado:** `permutation_importance`.


---

## Sugestão de auto-check formativo

Perguntas para **você** refletir sobre o que aprendeu — **não** são critérios de nota.

- [ ] Entendi o formato dos dados, as classes e se há desbalanceamento.
- [ ] Implementei treino/teste, escalonamento no treino e três modelos baseline no **meu** código.
- [ ] Explorei hiperparâmetros (árvore, SVM, NB) com validação no **treino** e anotei o que mudou ao variar parâmetros.
- [ ] Consigo explicar a diferença entre a Parte 3 (escolha de hiperparâmetros) e a Parte 4 (CV com pipeline no conjunto completo).
- [ ] Explico por que o `StandardScaler` deve ficar **dentro** do `Pipeline` na validação cruzada.
- [ ] (Opcional) Experimentei uma ou mais seções da Parte 5.

Se **reiniciar o kernel** e executar em ordem, a demonstração e os imports devem rodar; o restante depende do código que você for acrescentando.

**Bom trabalho!**
